# Text Generation via Speculative Decoding using FastDraft and OpenVINO™

As model sizes grow, Generative AI implementations require significant inference resources. This not only increases the cost per generation from a prompt, but also increases the power consumption used to serve such requests.

Inference optimizations for text generation are essential for reducing costs and power consumption. When optimizing the inference process, the amount of time and energy required to generate text can be significantly reduced. This can lead to cost savings in terms of hardware and software, as well as reduced power consumption. Additionally, inference optimizations can help improve the accuracy of text generation as well as the speed at which it can be generated. This can lead to an improved user experience and increased efficiency in text-generation tasks. In summary, inference optimizations for text generation are essential to reduce costs and power consumption, while also improving the accuracy and speed of text generation.


Speculative decoding (or [assisted-generation](https://huggingface.co/blog/assisted-generation#understanding-text-generation-latency)) is a recent technique, that allows to speed up token generation when an additional smaller draft model is used alongside with the main model.

Speculative decoding works the following way. The draft model predicts the next K tokens one by one in an autoregressive manner, while the main model validates these predictions and corrects them if necessary. We go through each predicted token, and if a difference is detected between the draft and main model, we stop and keep the last token predicted by the main model. Then the draft model gets the latest main prediction and again tries to predict the next K tokens, repeating the cycle.

This approach reduces the need for multiple infer requests to the main model, enhancing performance. For instance, in more predictable parts of text generation, the draft model can, in best-case scenarios, generate the next K tokens that exactly match the target. In that case they are validated in a single inference request to the main model (which is bigger, more accurate but slower) instead of running K subsequent requests. More details can be found in the original [paper](https://arxiv.org/pdf/2211.17192.pdf).

![](https://github.com/user-attachments/assets/eb999dea-d98b-42bb-835e-28d3054e1a84)

Possibility to achieve significant speedup with Speculative Decoding is highly depends on selection of a high-quality draft model that is both efficient and well-aligned with the target. FastDraft is a novel and efficient approach for pre-training and aligning a draft model to any LLM to be used with speculative decoding, by incorporating efficient pre-training followed by fine-tuning over synthetic datasets generated by the target model. FastDraft was presented in the [paper](https://arxiv.org/abs/2411.11055) at [ENLSP@NeurIPS24](https://neurips2024-enlsp.github.io/accepted_papers.html) by Intel Labs.

FastDraft pre-trained draft models achieve impressive results in key metrics of acceptance rate, block efficiency and up to 3x memory bound speed up
when evaluated on code completion and up to 2x in summarization, text completion and instruction tasks and unlock large language models inference on AI-PC and other edge-devices.

In this tutorial we consider how to apply Speculative decoding using FastDraft and OpenVINO GenAI.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/speculative-sampling/speculative-sampling.ipynb" />

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Prepare models](#Prepare-models)
    - [Select inference device](#Select-inference-device)
- [Run target model without speculative decoding](#Run-target-model-without-speculative-decoding)
- [Run Speculative decoding pipeline](#Run-Speculative-decoding-pipeline)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/speculative-sampling/speculative-sampling.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)


First, we should install the [OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) for running model inference.

![](https://media.githubusercontent.com/media/openvinotoolkit/openvino.genai/refs/heads/master/src/docs/openvino_genai.svg)

[OpenVINO™ GenAI](https://github.com/openvinotoolkit/openvino.genai) is a library of the most popular Generative AI model pipelines, optimized execution methods, and samples that run on top of highly performant [OpenVINO Runtime](https://github.com/openvinotoolkit/openvino).

This library is friendly to PC and laptop execution, and optimized for resource consumption. It requires no external dependencies to run generative models as it already includes all the core functionality (e.g. tokenization via openvino-tokenizers).


In [1]:
%pip install --pre -U openvino-genai --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly huggingface_hub datasets ipywidgets

Looking in indexes: https://pypi.org/simple, https://storage.openvinotoolkit.org/simple/wheels/nightly


  Using cached openvino_genai-2026.5.0.0.dev20261005-2735-cp312-cp312-manylinux_2_28_x86_64.whl (7.0 MB)


  Using cached huggingface_hub-2.1.1-py3-none-any.whl.metadata (16 kB)


  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)


  Using cached openvino_tokenizers-2026.5.0.0.dev20261005-py3-none-manylinux_2_28_x86_64.whl (1.9 MB)


...

  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 21/24 [aiohttp]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 22/24 [huggingface_hub]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 23/24 [datasets]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [datasets]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.22.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 2.1.1 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


## Prepare models
[back to top ⬆️](#Table-of-contents:)

As example, we will use already converted LLMs from [OpenVINO collection](https://huggingface.co/collections/OpenVINO/llm-6687aaa2abca3bbcec71a9bd).
You can find OpenVINO optimized FastDraft models can be found in this [collection](https://huggingface.co/collections/OpenVINO/speculative-decoding-draft-models-673f5d944d58b29ba6e94161). You can choose either Phi-3 or Phi-4 to be used for demonstrating FastDraft performance. For Phi-3 we will use [Phi-3-mini-4k-instruct-int4-ov](https://huggingface.co/OpenVINO/Phi-3-mini-4k-instruct-int4-ov) as target model and [Phi-3-mini-FastDraft-50M-int8-ov](https://huggingface.co/OpenVINO/Phi-3-mini-FastDraft-50M-int8-ov) as draft. For Phi-4 we will use [Phi-4-mini-instruct-int4-ov](https://huggingface.co/OpenVINO/Phi-4-mini-instruct-int4-ov) as target model and [Phi-4-mini-FastDraft-120M-int8-ov](https://huggingface.co/OpenVINO/Phi-4-mini-FastDraft-120M-int8-ov) as draft.

In case, if you want run own models, you should convert them using [Hugging Face Optimum](https://huggingface.co/docs/optimum/intel/openvino/export) library accelerated by OpenVINO integration. More details about model preparation can be found in [OpenVINO LLM inference guide](https://docs.openvino.ai/2025/openvino-workflow-generative/genai-model-preparation.html)

### Select model
[back to top ⬆️](#Table-of-contents:)

In [2]:
import ipywidgets as widgets

model = widgets.Dropdown(
    options=["Phi-3", "Phi-4"],
    value="Phi-3",  # default value
    description="Select Model:",
)

model

Dropdown(description='Select Model:', options=('Phi-3', 'Phi-4'), value='Phi-3')

### Download target and draft models

In [3]:
from pathlib import Path
import huggingface_hub as hf_hub

if model.value == "Phi-4":
    target_model_id = "OpenVINO/Phi-4-mini-instruct-int4-ov"
    draft_model_id = "OpenVINO/Phi-4-mini-FastDraft-120M-int8-ov"
elif model.value == "Phi-3":
    target_model_id = "OpenVINO/Phi-3-mini-4k-instruct-int4-ov"
    draft_model_id = "OpenVINO/Phi-3-mini-FastDraft-50M-int8-ov"
else:
    print(f"Model {model} is not supported in this demo.")

draft_model_path = Path(draft_model_id.split("/")[-1])
target_model_path = Path(target_model_id.split("/")[-1])

if not draft_model_path.exists():
    hf_hub.snapshot_download(draft_model_id, local_dir=draft_model_path)
if not target_model_path.exists():
    hf_hub.snapshot_download(target_model_id, local_dir=target_model_path)

### Select inference device
[back to top ⬆️](#Table-of-contents:)


Select the device from dropdown list for running inference using OpenVINO.
> **Note**: For achieving maximal performance, we recommend to use GPU as target device if it is available.

In [4]:
import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU", "AUTO"])
display(device)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("speculative-sampling.ipynb")

Dropdown(description='Device:', options=('CPU', 'GPU'), value='CPU')

## Run target model without speculative decoding
[back to top ⬆️](#Table-of-contents:)

OpenVINO GenAI provides easy-to-use API for running text generation. Firstly we will create pipeline with `LLMPipeline`. `LLMPipeline` is the main object used for decoding. You can construct it straight away from the folder with the converted model. It will automatically load the `main model`, `tokenizer`, `detokenizer` and default `generation configuration`. 
After that we will configure parameters for decoding. 
Then we just run `generate` method and get the output in text format. We do not need to encode input prompt according to model expected template or write post-processing code for logits decoder, it will be done easily with LLMPipeline. 

To obtain intermediate generation results without waiting until when generation is finished, we will write streamer function.

In [5]:
import openvino_genai as ov_genai
import time

print("Initializing Auto-Regressive pipeline...")
pipe = ov_genai.LLMPipeline(target_model_path, device.value)

config = ov_genai.GenerationConfig()
config.max_new_tokens = 200
config.apply_chat_template = False

prompt = '''<s>

def prime_fib(n: int):
    """
    prime_fib returns n-th number that is a Fibonacci number and it's also prime.
    >>> prime_fib(1)
    2
    >>> prime_fib(2)
    3
    >>> prime_fib(3)
    5
    >>> prime_fib(4)
    13
    >>> prime_fib(5)
    89
    """'''


def streamer(subword):
    print(subword, end="", flush=True)
    # Return flag corresponds whether generation should be stopped.
    # False means continue generation.
    return False


start_time = time.perf_counter()
pipe.generate(prompt, config, streamer=streamer)
end_time = time.perf_counter()

Initializing Auto-Regressive pipeline...
Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
Model: OV

Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
[INFO] PA backend is enabled.


 def

 is

_

...

 prime

 F

ib

on

acci number

In [6]:
import gc

print(f"Generation time: {end_time - start_time:.2f}s")
del pipe
gc.collect()

Generation time: 8.14s


0

## Run Speculative decoding pipeline
[back to top ⬆️](#Table-of-contents:)

To enable Speculative decoding in `LLMPipeline,` we should additionally provide the `draft_model` structure and `SchedulerConfig` for resource management. 

![](https://github.com/user-attachments/assets/69f5c096-abca-4f97-952b-291c52eb3444)


As shown in the figure above, speculative decoding works by splitting the generative process into two stages. In the first stage, a fast, but less accurate draft model (AKA assistant) autoregressively generates a sequence of tokens. In the second stage, a large, but more accurate target model conducts parallelized verification over the generated draft tokens. This process allows the target model to produce multiple tokens in a single forward pass and thus accelerate autoregressive decoding. The success of speculative decoding largely hinges on the speculation lookahead (SL), i.e. the number of tokens produced by the draft model in each iteration.  The straightforward method, based on [Leviathan et al.](https://arxiv.org/pdf/2211.17192), uses a static value of the speculation lookahead and involves generating a constant number of candidate tokens at each speculative iteration. You can adjust the number of candidates using `num_assistant_tokens` parameter in generation config. If the assistant model's confidence in its prediction for the current token is lower than this threshold, the assistant model stops the current token generation iteration is not yet reached.

In [7]:
draft_model = ov_genai.draft_model(draft_model_path, device.value)
config.num_assistant_tokens = 5

print("Initializing Speculative-Decoding pipeline...")
pipe = ov_genai.LLMPipeline(target_model_path, device.value, draft_model=draft_model)

start_time = time.perf_counter()
result = pipe.generate(prompt, config, streamer=streamer)
end_time = time.perf_counter()

Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: Initializing Speculative-Decoding pipeline...
PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
Mode

Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
Model: OV Detokenizer
  NETWORK_NAME: detokenizer


Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
Model: LLM with Paged Attention
  NETWORK

   def is_prime

(num):


...

-

th

 prime F


Generated tokens: 200
Total duration, sec: 4.15296
Draft model duration, sec: 0.466212
Main model duration, sec: 3.68675
Draft model duration, %: 11.226
Main model duration, %: 88.774
Token per sec: 48.1584
AVG acceptance rate, %: 50.5714
REQUEST_ID: 0
Main model iterations: 56
AVG acceptance rate, %: 50.5714
Generated tokens by draft model: 278
Accepted token rate, %: 50.7194
Request_id: 0 ||| 20 80 60 60 100 20 100 100 20 100 100 100 60 40 0 60 80 100 80 80 100 100 0 80 100 100 100 20 100 0 20 60 100 0 0 60 80 100 100 0 20 40 80 0 40 0 0 0 0 20 20 0 0 0 0 33.3333 
ibon

acci number

In [8]:
print(f"Generation time: {end_time - start_time:.2f}s")

Generation time: 4.16s


Alternative approach, Dynamic Speculative Decoding, described in the [paper](https://arxiv.org/abs/2405.04304) is based on heuristics and adjusts the number of candidate tokens for the next iteration based on the acceptance rate of the current iteration. If all speculative tokens are correct, the number of candidate tokens increases; otherwise, it decreases. For adjusting number of tokens `assistant_confidence_threshold` parameters should be used. If the assistant model's confidence in its prediction for the current token is lower than this threshold, the assistant model stops the current token generation iteration, even if the number of `num_assistant_tokens` is not yet reached.  You can find more details in this [blog post](https://huggingface.co/blog/dynamic_speculation_lookahead). This approach has advantages for cases, when optimal number of tokens for draft model is unknown and draft model has low acceptance rate.

>*Note*: For small and fast draft models like FastDraft, you may not see benefit for dynamic speculative decoding.

In [9]:
config.num_assistant_tokens = 0
config.assistant_confidence_threshold = 0.05

start_time = time.perf_counter()
result = pipe.generate(prompt, config, streamer)
end_time = time.perf_counter()

   def

 is_

prime

(num):


       if

...

_f

ib(

3)


Generated tokens: 200
Total duration, sec: 3.95776
Draft model duration, sec: 0.296587
Main model duration, sec: 3.66117
Draft model duration, %: 7.49382
Main model duration, %: 92.5062
Token per sec: 50.5337
AVG acceptance rate, %: 86.7924
REQUEST_ID: 0
Main model iterations: 106
AVG acceptance rate, %: 86.7924
Generated tokens by draft model: 106
Accepted token rate, %: 86.7924
Request_id: 0 ||| 100 100 100 0 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 100 0 0 100 100 100 100 0 100 100 100 100 100 0 100 100 0 100 100 100 100 100 100 0 100 100 0 100 100 100 100 100 100 100 100 100 100 100 100 100 0 100 100 100 100 100 100 0 0 100 100 0 100 100 100 100 100 100 100 100 0 100 100 0 100 100 100 100 100 100 100 100 100 100 100 100 



prime_

In [10]:
print(f"Generation time: {end_time - start_time:.2f}s")

Generation time: 3.97s


## Evaluate Speculative Decoding on multiple examples

Configure the data type and the number of examples to run:

In [11]:
num_samples_to_select = 50

import ipywidgets as widgets

data_options = ["Code", "Text"]
data_type = widgets.Dropdown(
    options=data_options,
    value=data_options[0],
    description="Data type:",
    disabled=False,
)
data_type

Dropdown(description='Data type:', options=('Code', 'Text'), value='Code')

Load the dataset and prepare the prompts:

In [12]:
from datasets import load_dataset

print("loading dataset...")

if data_type.value == "Code":
    ds = load_dataset("openai/openai_humaneval", split="test")
    prompts = ds["prompt"]
    prompts = ["<s>" + prompts[i] for i in range(num_samples_to_select)]
else:
    ds = load_dataset("abisee/cnn_dailymail", "3.0.0", split="test")
    prompts = ds["article"]
    prompts = [
        "<|user|> ###\nArticle: " + prompts[i] + "\n\nSummarize the above article in 5 sentence.\n<|end|><|assistant|>" for i in range(num_samples_to_select)
    ]
print("Done")

loading dataset...


Done


Run auto-regressive generation and get total runtime per example:

In [13]:
import openvino_genai as ov_genai
import time
from tqdm import tqdm

print("Initializing Auto-Regressive pipline...")
pipe = ov_genai.LLMPipeline(target_model_path, device.value)

config = ov_genai.GenerationConfig()
config.apply_chat_template = False
config.max_new_tokens = 330
if data_type.value == "Code":
    config.max_new_tokens = 128

# warmup
print("Running Warmup...")
for i in range(10):
    pipe.generate("this is a warmup prompt", config)

print("Running Auto-Regressive generation...")
times_auto_regressive = []
for prompt in tqdm(prompts):
    start_time = time.perf_counter()
    result = pipe.generate(prompt, config)
    end_time = time.perf_counter()
    times_auto_regressive.append(end_time - start_time)
print("Done")

import gc

del pipe
gc.collect()

Initializing Auto-Regressive pipline...
Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
Model: OV 

Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
[INFO] PA backend is enabled.
Running War

Running Auto-Regressive generation...


  0%|          | 0/50 [00:00<?, ?it/s]

  2%|▏         | 1/50 [00:02<02:20,  2.86s/it]

...

 98%|█████████▊| 49/50 [02:31<00:03,  3.11s/it]

100%|██████████| 50/50 [02:35<00:00,  3.43s/it]

100%|██████████| 50/50 [02:35<00:00,  3.12s/it]

Done


272

Now run generation with speculative-decoding:

In [14]:
config.num_assistant_tokens = 5
draft_model = ov_genai.draft_model(draft_model_path, device.value)

print("Initializing Speculative-Decoding pipline...")
pipe = ov_genai.LLMPipeline(target_model_path, device.value, draft_model=draft_model)

# warmup
print("Running Warmup...")
for i in range(10):
    pipe.generate("this is a warmup prompt", config)

times_speculative_decoding = []
print("Running Speculative Decoding generation...")
for prompt in tqdm(prompts):
    start_time = time.perf_counter()
    result = pipe.generate(prompt, config)
    end_time = time.perf_counter()
    times_speculative_decoding.append((end_time - start_time))
print("Done")

Model: OV Tokenizer
Initializing Speculative-Decoding pipline...
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
Model

Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
Model: LLM with Paged Attention
  NETWORK


Generated tokens: 128
Total duration, sec: 5.05358
Draft model duration, sec: 0.644401
Main model duration, sec: 4.40918
Draft model duration, %: 12.7514
Main model duration, %: 87.2486
Token per sec: 25.3286
AVG acceptance rate, %: 9.75
REQUEST_ID: 0
Main model iterations: 84
AVG acceptance rate, %: 9.75
Generated tokens by draft model: 415
Accepted token rate, %: 9.87952
Request_id: 0 ||| 0 20 0 0 0 0 0 20 0 0 0 0 20 0 20 20 20 0 20 0 20 20 0 0 0 20 0 0 20 0 20 0 0 0 0 0 0 0 0 0 0 20 40 0 0 60 0 40 0 20 0 0 0 40 0 40 0 20 20 0 0 0 20 20 0 0 20 0 0 0 40 0 0 60 60 20 0 40 0 0 0 0 0 0 



Generated tokens: 128
Total duration, sec: 3.72733
Draft model duration, sec: 0.508638
Main model duration, sec: 3.21869
Draft model duration, %: 13.6462
Main model duration, %: 86.3538
Token per sec: 34.341
AVG acceptance rate, %: 9.75
REQUEST_ID: 0
Main model iterations: 84
AVG acceptance rate, %: 9.75
Generated tokens by draft model: 415
Accepted token rate, %: 9.87952
Request_id: 0 ||| 0 20 0 0 0 0 0 20 0 0 0 0 20 0 20 20 20 0 20 0 20 20 0 0 0 20 0 0 20 0 20 0 0 0 0 0 0 0 0 0 0 20 40 0 0 60 0 40 0 20 0 0 0 40 0 40 0 20 20 0 0 0 20 20 0 0 20 0 0 0 40 0 0 60 60 20 0 40 0 0 0 0 0 0 



Generated tokens: 128
Total duration, sec: 3.7204
Draft model duration, sec: 0.508893
Main model duration, sec: 3.21151
Draft model duration, %: 13.6784
Main model duration, %: 86.3215
Token per sec: 34.4049
AVG acceptance rate, %: 9.75
REQUEST_ID: 0
Main model iterations: 84
AVG acceptance rate, %: 9.75
Generated tokens by draft model: 415
Accepted token rate, %: 9.87952
Request_id: 0 ||| 0 20 0 0 0 0 0 20 0 0 0 0 20 0 20 20 20 0 20 0 20 20 0 0 0 20 0 0 20 0 20 0 0 0 0 0 0 0 0 0 0 20 40 0 0 60 0 40 0 20 0 0 0 40 0 40 0 20 20 0 0 0 20 20 0 0 20 0 0 0 40 0 0 60 60 20 0 40 0 0 0 0 0 0 


...

 98%|█████████▊| 49/50 [01:13<00:01,  1.60s/it]


Generated tokens: 41
Total duration, sec: 0.740626
Draft model duration, sec: 0.087885
Main model duration, sec: 0.652741
Draft model duration, %: 11.8663
Main model duration, %: 88.1337
Token per sec: 55.3586
AVG acceptance rate, %: 41.5833
REQUEST_ID: 0
Main model iterations: 12
AVG acceptance rate, %: 41.5833
Generated tokens by draft model: 65
Accepted token rate, %: 38.4615
Request_id: 0 ||| 40 20 100 20 100 100 0 20 20 20 60 0 


100%|██████████| 50/50 [01:16<00:00,  1.78s/it]

100%|██████████| 50/50 [01:16<00:00,  1.52s/it]


Generated tokens: 128
Total duration, sec: 2.21269
Draft model duration, sec: 0.296617
Main model duration, sec: 1.91607
Draft model duration, %: 13.4053
Main model duration, %: 86.5947
Token per sec: 57.8482
AVG acceptance rate, %: 37.7674
REQUEST_ID: 0
Main model iterations: 43
AVG acceptance rate, %: 37.7674
Generated tokens by draft model: 216
Accepted token rate, %: 37.5
Request_id: 0 ||| 0 0 20 20 0 80 0 60 20 0 40 100 40 20 0 100 100 60 40 60 100 100 100 60 0 20 0 40 100 80 0 100 40 20 0 40 20 20 0 0 0 0 25 
Done


Now let's calculate the speedup:

In [15]:
avg_speedup = sum([x / y for x, y in zip(times_auto_regressive, times_speculative_decoding)]) / len(prompts)
print(f"average speedup: {avg_speedup:.2f}")

average speedup: 2.07
